In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = pd.read_csv("Advertising.csv")

print("FIRST 5 ROWS")
print(data.head())

print("\nORIGINAL COLUMN NAMES")
print(data.columns.tolist())

data.columns = data.columns.astype(str).str.strip()

data = data.loc[:, ~data.columns.str.lower().str.startswith("unnamed")]

column_map = {}

for column in data.columns:
    clean_column = column.strip().lower()

    if clean_column == "tv":
        column_map[column] = "TV"
    elif clean_column == "radio":
        column_map[column] = "Radio"
    elif clean_column == "newspaper":
        column_map[column] = "Newspaper"
    elif clean_column == "sales":
        column_map[column] = "Sales"

data = data.rename(columns=column_map)

print("\nCLEANED COLUMN NAMES")
print(data.columns.tolist())

required_columns = ["TV", "Radio", "Newspaper", "Sales"]

missing_columns = [column for column in required_columns if column not in data.columns]

if len(missing_columns) > 0:
    print("\nMissing columns:", missing_columns)
    print("Available columns:", data.columns.tolist())
    raise ValueError("Required columns are missing from Advertising.csv")

data = data[required_columns]

print("\nDATASET INFORMATION")
print(data.info())

print("\nDATASET SHAPE")
print(data.shape)

print("\nMISSING VALUES")
print(data.isnull().sum())

print("\nDUPLICATE ROWS")
print(data.duplicated().sum())

data = data.drop_duplicates()

print("\nSHAPE AFTER REMOVING DUPLICATES")
print(data.shape)

print("\nSUMMARY STATISTICS")
print(data.describe())

print("\nFINAL COLUMN NAMES")
print(data.columns.tolist())

plt.figure(figsize=(8, 5))
sns.histplot(data["Sales"], bins=30, kde=True)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="TV", y="Sales")
plt.title("TV Advertising vs Sales")
plt.xlabel("TV Advertising")
plt.ylabel("Sales")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="Radio", y="Sales")
plt.title("Radio Advertising vs Sales")
plt.xlabel("Radio Advertising")
plt.ylabel("Sales")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="Newspaper", y="Sales")
plt.title("Newspaper Advertising vs Sales")
plt.xlabel("Newspaper Advertising")
plt.ylabel("Sales")
plt.show()

plt.figure(figsize=(8, 5))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

X = data[["TV", "Radio", "Newspaper"]]
y = data["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTRAINING DATA SIZE")
print(X_train.shape)

print("\nTESTING DATA SIZE")
print(X_test.shape)

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_prediction = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_prediction)
linear_mse = mean_squared_error(y_test, linear_prediction)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_prediction)

print("\nLINEAR REGRESSION RESULTS")
print("MAE:", linear_mae)
print("MSE:", linear_mse)
print("RMSE:", linear_rmse)
print("R2 Score:", linear_r2)

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

random_forest_model.fit(X_train, y_train)

random_forest_prediction = random_forest_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, random_forest_prediction)
rf_mse = mean_squared_error(y_test, random_forest_prediction)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, random_forest_prediction)

print("\nRANDOM FOREST RESULTS")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)
print("R2 Score:", rf_r2)

gradient_model = GradientBoostingRegressor(
    random_state=42
)

gradient_model.fit(X_train, y_train)

gradient_prediction = gradient_model.predict(X_test)

gradient_mae = mean_absolute_error(y_test, gradient_prediction)
gradient_mse = mean_squared_error(y_test, gradient_prediction)
gradient_rmse = np.sqrt(gradient_mse)
gradient_r2 = r2_score(y_test, gradient_prediction)

print("\nGRADIENT BOOSTING RESULTS")
print("MAE:", gradient_mae)
print("MSE:", gradient_mse)
print("RMSE:", gradient_rmse)
print("R2 Score:", gradient_r2)

model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        linear_mae,
        rf_mae,
        gradient_mae
    ],
    "MSE": [
        linear_mse,
        rf_mse,
        gradient_mse
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse,
        gradient_rmse
    ],
    "R2 Score": [
        linear_r2,
        rf_r2,
        gradient_r2
    ]
})

print("\nMODEL COMPARISON")
print(model_results)

best_model_index = model_results["R2 Score"].idxmax()

best_model_name = model_results.loc[best_model_index, "Model"]

print("\nBEST MODEL")
print(best_model_name)

feature_importance = random_forest_model.feature_importances_

feature_importance_data = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importance
})

feature_importance_data = feature_importance_data.sort_values(
    by="Importance",
    ascending=False
)

print("\nFEATURE IMPORTANCE")
print(feature_importance_data)

most_important_feature = feature_importance_data.iloc[0]["Feature"]

plt.figure(figsize=(8, 5))
sns.barplot(
    data=feature_importance_data,
    x="Importance",
    y="Feature"
)
plt.title("Advertising Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Advertising Channel")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(y_test, random_forest_prediction)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual Sales vs Predicted Sales")

minimum_value = min(y_test.min(), random_forest_prediction.min())
maximum_value = max(y_test.max(), random_forest_prediction.max())

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value]
)

plt.show()

new_data = pd.DataFrame({
    "TV": [150],
    "Radio": [30],
    "Newspaper": [20]
})

new_prediction = random_forest_model.predict(new_data)

print("\nNEW SALES PREDICTION")
print("TV Advertising:", new_data["TV"].iloc[0])
print("Radio Advertising:", new_data["Radio"].iloc[0])
print("Newspaper Advertising:", new_data["Newspaper"].iloc[0])
print("Predicted Sales:", new_prediction[0])

print("\nFINAL PROJECT SUMMARY")
print("Number of records:", len(data))
print("Number of features:", len(X.columns))
print("Best Model:", best_model_name)
print("Linear Regression R2:", linear_r2)
print("Random Forest R2:", rf_r2)
print("Gradient Boosting R2:", gradient_r2)
print("Most Important Advertising Channel:", most_important_feature)

print("\nPROJECT COMPLETED SUCCESSFULLY")